# Day 17 — Scaled dot-product attention  ·  Milestone 2 capstone 🏁

**Milestone 2 finale:** assemble every piece — $QK^\top$ (day 11), $\div\sqrt{d}$ (day 16), softmax over keys (days 13–14), weighted average of values (day 15) — into the mechanism at the heart of every Transformer.

## 1. Review (~5 min)

Recall **day 13** (softmax) and **day 15** (weighted average).

In [ ]:
import numpy as np
rng = np.random.default_rng(17)

**R1.** (day 13) softmax.

In [ ]:
def r_softmax(s):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_r1(fn):
    s = rng.normal(size=6); e = np.exp(s - s.max())
    assert np.allclose(np.asarray(fn(s)), e / e.sum())
    print("✅ Review R1 passed")

check_r1(r_softmax)

**R2.** (day 15) weighted average $\sum_i w_i V_i = w^\top V$.

In [ ]:
def r_weighted_average(w, V):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_r2(fn):
    w = rng.random(4); w = w / w.sum(); V = rng.normal(size=(4, 3))
    assert np.allclose(np.asarray(fn(w, V)), w @ V)
    print("✅ Review R2 passed")

check_r2(r_weighted_average)

## 2. Lecture note (~10 min): putting it all together

**The itch.** For each **query**, we want to look over a set of **keys**, decide how much to attend to each, and return a blend of the corresponding **values**. Every ingredient is now in hand.

**The picture & formula.** With queries $Q$ $(n_q, d)$, keys $K$ $(n_k, d)$, values $V$ $(n_k, d_v)$:
$$\operatorname{Attention}(Q,K,V) = \operatorname{softmax}\!\Big(\frac{QK^\top}{\sqrt{d}}\Big)\,V.$$
Step by step: (1) $QK^\top$ — all query–key similarities (day 11); (2) $\div\sqrt{d}$ — keep them sane (day 16); (3) **softmax along each row** (over the keys) — turn each query's scores into attention weights that sum to 1 (days 13–14); (4) multiply by $V$ — each query's output is a weighted average of value vectors (day 15). Output shape: $(n_q, d_v)$.

**Knobs & effect.** The softmax is **row-wise** (`axis=-1`): query $i$'s weights live in row $i$ and sum to 1. Each output row is a convex blend of the value rows — the query *retrieves* a mixture of values, weighted by similarity to the keys.

**In the wild.** This single equation, stacked into multi-head layers, is the engine of the Transformer (Vaswani et al. 2017) and therefore of essentially every modern LLM.

## 3. Exercises (~15 min)

### Exercise 1 — row-wise softmax
Apply softmax **independently to each row** of a matrix `S` (shape `(m, n)`); every row of the output sums to 1. (This is the attention-weights step.)

In [ ]:
def softmax_rows(S):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e1(fn):
    S = rng.normal(size=(4, 5))
    W = np.asarray(fn(S)); assert W.shape == S.shape
    assert np.allclose(W.sum(axis=1), np.ones(4))
    for i in range(4):
        e = np.exp(S[i] - S[i].max())
        assert np.allclose(W[i], e / e.sum())
    print("✅ Exercise 1 passed")

check_e1(softmax_rows)

### Exercise 2 — scaled dot-product attention (capstone 🏁)
Implement $\operatorname{softmax}(QK^\top/\sqrt{d})\,V$. (Checked against a step-by-step oracle; output shape `(n_q, d_v)`; each weight row sums to 1.)

In [ ]:
def attention(Q, K, V):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def attn_oracle(Q, K, V):
    d = Q.shape[1]; S = Q @ K.T / np.sqrt(d)
    S = S - S.max(axis=1, keepdims=True); E = np.exp(S); W = E / E.sum(axis=1, keepdims=True)
    return W @ V

def check_e2(fn):
    nq, nk, d, dv = 3, 5, 4, 6
    Q = rng.normal(size=(nq, d)); K = rng.normal(size=(nk, d)); V = rng.normal(size=(nk, dv))
    out = np.asarray(fn(Q, K, V)); assert out.shape == (nq, dv)
    assert np.allclose(out, attn_oracle(Q, K, V))
    print("✅ Exercise 2 passed")

check_e2(attention)

### Exercise 3 — attention *retrieves* (the effect)
When one key is strongly aligned with a query, that key dominates the softmax and the output ≈ **that key's value**. The check plants such a key and expects the matching value back — attention as soft lookup.

In [ ]:
def attend_one(q, K, V):
    # q: shape (d,). Return the attention output for this single query (shape (d_v,)).
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e3(fn):
    d, dv = 4, 3
    q = rng.normal(size=d)
    K = rng.normal(size=(5, d)); K[2] = 12.0 * q / np.linalg.norm(q)  # key 2 strongly aligned
    V = rng.normal(size=(5, dv))
    out = np.asarray(fn(q, K, V))
    assert out.shape == (dv,)
    assert np.allclose(out, V[2], atol=1e-2), "should retrieve value of the aligned key"
    print("✅ Exercise 3 passed")

check_e3(attend_one)

## Solutions (collapsed — peek only if stuck)

`ref_`-prefixed answers. Running the next cell validates everything against the checks above.

In [ ]:
def ref_softmax(s):
    e = np.exp(s - np.max(s)); return e / e.sum()

def ref_weighted_average(w, V):
    return w @ V

def ref_softmax_rows(S):
    E = np.exp(S - S.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True)

def ref_attention(Q, K, V):
    d = Q.shape[1]
    return ref_softmax_rows(Q @ K.T / np.sqrt(d)) @ V

def ref_attend_one(q, K, V):
    return ref_attention(q[None, :], K, V)[0]

check_r1(ref_softmax)
check_r2(ref_weighted_average)
check_e1(ref_softmax_rows)
check_e2(ref_attention)
check_e3(ref_attend_one)
print("\nAll reference solutions pass. 🎉  You built attention!  One more day to tie it to cosine similarity.")